In [ ]:
# ==============================================================================
# ASTRO PLANNER V2 (CORRECTED LOGIC)
# ==============================================================================

# --- 1. IMPORTS ---
APP_BUILD = "2026-06-01-lightpollution-v1"

import os
import requests
import time
from datetime import datetime, timezone, timedelta
from uuid import NAMESPACE_URL, uuid5
from zoneinfo import ZoneInfo, ZoneInfoNotFoundError
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
import ipywidgets as widgets
from IPython.display import display, clear_output
from skyfield.api import Topos, load, Star
from skyfield.almanac import fraction_illuminated

try:
    from scipy.interpolate import PchipInterpolator
except ImportError:
    PchipInterpolator = None

# --- 2. CORE AND PLANET INITIALIZATION ---
print("Initializing the JPL astronomy kernel (this may take a few seconds)...")
planets = load('de421.bsp')
earth, sun, moon = planets['earth'], planets['sun'], planets['moon']
ts = load.timescale()

solar_system_dict = {
    'Sun': sun,
    'Moon': moon,
    'Mercury': planets['mercury'],
    'Venus': planets['venus'],
    'Mars': planets['mars'],
    'Jupiter': planets['jupiter barycenter'],
    'Saturn': planets['saturn barycenter'],
    'Uranus': planets['uranus barycenter'],
    'Neptune': planets['neptune barycenter']
}

# --- 3. CATALOG LOADING AND DECODING ---
def load_catalog():
    url = "https://raw.githubusercontent.com/mattiaverga/OpenNGC/master/database_files/NGC.csv"
    cache = "NGC_full_decoded.csv"
    
    if os.path.exists(cache):
        df = pd.read_csv(cache, sep=';', low_memory=False)
    else:
        print("Downloading the full OpenNGC catalog (~13,000 objects)...")
        df = pd.read_csv(url, sep=';', low_memory=False)
        df.to_csv(cache, sep=';', index=False)
    
    type_map = {
        'G': 'Galaxy', 'PN': 'Planetary nebula', 'OC': 'Open cluster',
        'GC': 'Globular cluster', 'Nb': 'Nebula', 'C+N': 'Cluster + nebula',
        'Ast': 'Asterism', 'Kt': 'Knot', '*': 'Star', 'D*': 'Double star', 
        '***': 'Triple star', 'SC': 'Star cloud', 'HII': 'HII region'
    }

    def format_name(row):
        m_val = str(row.get('M', '')).strip()
        t_val = str(row.get('Type', '')).strip()
        obj_type = type_map.get(t_val, 'DSO')
        name = str(row['Name'])
        
        if m_val and m_val.lower() != 'nan':
            try:
                return f"M{int(float(m_val))} ({name}) - {obj_type}"
            except (TypeError, ValueError):
                pass
        return f"{name} - {obj_type}"
        
    df['DisplayName'] = df.apply(format_name, axis=1)
    
    def dec_to_float(d):
        try:
            p = str(d).replace(':', ' ').split()
            sign = -1 if p[0].startswith('-') else 1
            value = sign * (abs(float(p[0])) + float(p[1])/60 + (float(p[2])/3600 if len(p)>2 else 0))
            return value if -90 <= value <= 90 else np.nan
        except (IndexError, TypeError, ValueError):
            return np.nan

    def ra_to_float(r):
        try:
            p = list(map(float, str(r).replace(':', ' ').split()))
            value = p[0] + p[1]/60 + (p[2]/3600 if len(p)>2 else 0)
            return value if 0 <= value < 24 else np.nan
        except (IndexError, TypeError, ValueError):
            return np.nan

    df['DecFloat'] = df['Dec'].apply(dec_to_float)
    df['RaFloat'] = df['RA'].apply(ra_to_float)
    cleaned = df[['DisplayName', 'RaFloat', 'DecFloat']].dropna().copy()
    cleaned.columns = ['Name', 'RaFloat', 'DecFloat']
    return cleaned.drop_duplicates(subset='Name').reset_index(drop=True)

print("Preparing the catalog...")
dso_catalog = load_catalog()

# --- 4. WEATHER DATA ---
def get_24h_weather(lat, lon):
    url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&hourly=cloud_cover,cloud_cover_low,cloud_cover_mid,cloud_cover_high,precipitation_probability,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_gusts_10m&wind_speed_unit=ms&timezone=auto&forecast_days=2"
    
    response = requests.get(url, timeout=15)
    if response.status_code != 200:
        raise RuntimeError(f"Open-Meteo API error (status {response.status_code}): {response.text}")

    try:
        payload = response.json()
        res = payload['hourly']
        forecast_timezone = ZoneInfo(payload['timezone'])
    except ZoneInfoNotFoundError as exc:
        raise RuntimeError("Open-Meteo returned an unknown time zone.") from exc
    except (KeyError, TypeError, ValueError) as exc:
        raise RuntimeError("Open-Meteo returned a response without an hourly forecast.") from exc
    
    df = pd.DataFrame({
        'time': pd.to_datetime(res['time']).tz_localize(forecast_timezone, ambiguous='infer', nonexistent='shift_forward').tz_convert(timezone.utc),
        'cloud_cover': res['cloud_cover'],
        'cloud_low': res['cloud_cover_low'],
        'cloud_mid': res['cloud_cover_mid'],
        'cloud_high': res['cloud_cover_high'],
        'precip_prob': res['precipitation_probability'],
        'temp': res['temperature_2m'],
        'humidity': res['relative_humidity_2m'],
        'wind': res['wind_speed_10m'],
        'gusts': res['wind_gusts_10m']
    })

    weather_columns = ['cloud_cover', 'cloud_low', 'cloud_mid', 'cloud_high', 'precip_prob', 'temp', 'humidity', 'wind', 'gusts']
    df[weather_columns] = df[weather_columns].apply(pd.to_numeric, errors='coerce')
    if df[weather_columns].isna().any().any():
        raise RuntimeError("Open-Meteo returned incomplete weather data.")
    
    now_utc = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0)
    end_utc = now_utc + timedelta(hours=24)
    forecast = df[(df['time'] >= now_utc) & (df['time'] < end_utc)].copy()
    if len(forecast) != 24:
        raise RuntimeError(f"Expected 24 forecast hours, received: {len(forecast)}.")
    forecast = forecast.reset_index(drop=True)
    forecast.attrs['display_timezone'] = forecast_timezone.key
    return forecast

# --- 5. MATH, SCORING, AND ICS GENERATION ---
def validate_inputs(lat, lon, sun_limit, min_alt):
    if not -90 <= lat <= 90:
        raise ValueError("Latitude must be between -90° and 90°.")
    if not -180 <= lon <= 180:
        raise ValueError("Longitude must be between -180° and 180°.")
    if not -90 <= sun_limit <= 90:
        raise ValueError("The Sun altitude limit must be between -90° and 90°.")
    if not 0 <= min_alt < 90:
        raise ValueError("The minimum target altitude must be between 0° and 90°.")

def moon_interference_percent(moon_phase, moon_altitude, separation):
    if moon_altitude <= 0:
        return 0.0
    interference = moon_phase * (moon_altitude / 90.0) * (1.0 - min(180, separation) / 180.0)
    return max(0.0, min(100.0, interference * 1.5 * 100.0))

def calculate_scoring(lat, lon, df_weather, targets, sun_limit, min_alt):
    validate_inputs(lat, lon, sun_limit, min_alt)
    observer = earth + Topos(latitude_degrees=lat, longitude_degrees=lon)
    results = {}
    
    for obj_name in targets:
        is_planet = False
        is_sun = False
        is_moon = False
        
        if obj_name in solar_system_dict:
            target = solar_system_dict[obj_name]
            is_planet = True
            if 'Sun' in obj_name: is_sun = True
            if 'Moon' in obj_name: is_moon = True
        else:
            obj_row = dso_catalog[dso_catalog['Name'] == obj_name].iloc[0]
            target = Star(ra_hours=obj_row['RaFloat'], dec_degrees=obj_row['DecFloat'])
        
        probs, reasons, altitudes = [], [], []
        
        for _, row in df_weather.iterrows():
            t_utc = ts.from_datetime(row['time'])
            
            # Ephemerides
            sun_alt, _, _ = observer.at(t_utc).observe(sun).apparent().altaz()
            obj_pos = observer.at(t_utc).observe(target).apparent()
            obj_alt, _, _ = obj_pos.altaz()
            altitudes.append(obj_alt.degrees)
            
            # 1. Hard altitude and Sun filters
            if is_sun:
                if obj_alt.degrees < min_alt:
                    probs.append(0); reasons.append("Sun is below the horizon")
                    continue
            else:
                if obj_alt.degrees < min_alt:
                    probs.append(0); reasons.append(f"Below {min_alt}°")
                    continue
                current_sun_limit = 0 if is_planet else sun_limit
                if sun_alt.degrees > current_sun_limit:
                    probs.append(0); reasons.append("Sky too bright")
                    continue
                    
            # 2. Soft penalties (weather)
            alt_factor = (obj_alt.degrees - min_alt) / (90.0 - min_alt)
            clear_sky = 1.0 - (row['cloud_cover'] / 100.0)
            base_p = clear_sky * (0.6 + 0.4 * alt_factor)
            
            # Precipitation penalty (smooth multiplier instead of a hard cutoff)
            precip_factor = 1.0 - (row['precip_prob'] / 100.0)
            
            # Wind penalty (gusts)
            wind_factor = max(0.0, 1.0 - max(0.0, row['gusts'] - 5) / 10.0)
            
            # 3. Moonlight penalty (DSO only)
            moon_factor = 1.0
            if not is_planet and not is_sun and not is_moon:
                moon_pos = observer.at(t_utc).observe(moon).apparent()
                moon_alt, _, _ = moon_pos.altaz()
                
                # Moonlight matters only while the Moon is above the horizon
                if moon_alt.degrees > 0:
                    moon_phase = fraction_illuminated(planets, 'moon', t_utc)
                    separation = moon_pos.separation_from(obj_pos).degrees
                    moon_factor = 1.0 - moon_interference_percent(moon_phase, moon_alt.degrees, separation) / 100.0

            # Final probability
            p = base_p * precip_factor * wind_factor * moon_factor * 100
            p = int(max(0, min(100, p)))
            
            # Build the matrix reason using issue priority
            if p < 30:
                if row['cloud_cover'] > 70: reasons.append("Cloudy")
                elif row['precip_prob'] > 20: reasons.append("Precipitation")
                elif row['gusts'] > 8: reasons.append("Wind")
                elif moon_factor < 0.6: reasons.append("Moonlight interference")
                else: reasons.append("Low altitude / poor conditions")
            else: 
                reasons.append("OK")
                
            probs.append(p)
            
        results[obj_name] = {'p': probs, 'r': reasons, 'alt': altitudes}
    return results

def calculate_light_pollution_diagnostics(lat, lon, df_weather, targets):
    observer = earth + Topos(latitude_degrees=lat, longitude_degrees=lon)
    dso_targets = [obj for obj in targets if obj not in solar_system_dict]
    stars = {}
    for obj in dso_targets:
        obj_row = dso_catalog[dso_catalog['Name'] == obj].iloc[0]
        stars[obj] = Star(ra_hours=obj_row['RaFloat'], dec_degrees=obj_row['DecFloat'])

    diagnostics = {
        'sun_alt': [],
        'moon_alt': [],
        'moon_phase': [],
        'objects': {
            obj: {'moon_separation': [], 'moon_risk': []}
            for obj in dso_targets
        }
    }

    for _, row in df_weather.iterrows():
        t_utc = ts.from_datetime(row['time'])
        observer_at_time = observer.at(t_utc)
        sun_alt, _, _ = observer_at_time.observe(sun).apparent().altaz()
        moon_pos = observer_at_time.observe(moon).apparent()
        moon_alt, _, _ = moon_pos.altaz()
        moon_phase = fraction_illuminated(planets, 'moon', t_utc)

        diagnostics['sun_alt'].append(sun_alt.degrees)
        diagnostics['moon_alt'].append(moon_alt.degrees)
        diagnostics['moon_phase'].append(moon_phase * 100.0)

        for obj, star in stars.items():
            obj_pos = observer_at_time.observe(star).apparent()
            separation = moon_pos.separation_from(obj_pos).degrees
            diagnostics['objects'][obj]['moon_separation'].append(separation)
            diagnostics['objects'][obj]['moon_risk'].append(
                moon_interference_percent(moon_phase, moon_alt.degrees, separation)
            )

    return diagnostics

def interpolate_for_chart(hours_local, values, clip=None):
    source_x = np.arange(len(values), dtype=float)
    chart_x = np.linspace(0.0, len(values) - 1, (len(values) - 1) * 6 + 1)
    source_values = np.asarray(values, dtype=float)
    if PchipInterpolator is not None and len(values) >= 2:
        chart_values = PchipInterpolator(source_x, source_values)(chart_x)
    else:
        chart_values = np.interp(chart_x, source_x, source_values)
    if clip is not None:
        chart_values = np.clip(chart_values, clip[0], clip[1])
    chart_times = pd.DatetimeIndex(hours_local.iloc[0] + pd.to_timedelta(chart_x, unit='h'))
    return chart_times, chart_values

def short_object_name(obj, max_length=30):
    return obj[:max_length] + '...' if len(obj) > max_length else obj

def add_solar_background(ax, chart_times, sun_altitudes):
    periods = [
        (sun_altitudes > 0, '#fff3a3', 'Day'),
        ((sun_altitudes <= 0) & (sun_altitudes > -6), '#ffd2a3', 'Civil twilight'),
        ((sun_altitudes <= -6) & (sun_altitudes > -12), '#b9d8ff', 'Nautical twilight'),
        ((sun_altitudes <= -12) & (sun_altitudes > -18), '#aeb8e8', 'Astronomical twilight'),
        (sun_altitudes <= -18, '#20294f', 'Dark sky'),
    ]
    for mask, color, _ in periods:
        ax.fill_between(chart_times, -95, 95, where=mask, color=color, alpha=0.22)
    return [Patch(facecolor=color, alpha=0.22, label=label) for _, color, label in periods]

def add_solar_boundaries(ax, sun_limit):
    boundaries = [
        (0, 'Horizon / Sun 0°', '-', 'black'),
        (-6, 'Sun -6°', ':', '#d97706'),
        (-12, 'Sun -12°', ':', '#2563eb'),
        (-18, 'Sun -18°', ':', '#4338ca'),
    ]
    matched_limit = False
    for altitude, label, linestyle, color in boundaries:
        if np.isclose(sun_limit, altitude):
            label += ' | your limit'
            matched_limit = True
        ax.axhline(altitude, color=color, linestyle=linestyle, linewidth=1, label=label)
    if not matched_limit:
        ax.axhline(sun_limit, color='#dc2626', linestyle='-.', linewidth=1.3, label=f'Your Sun limit {sun_limit:g}°')

def plot_light_pollution_chart(hours_local, results, diagnostics, targets, sun_limit, current_local, object_filter=None):
    chart_targets = targets if object_filter is None else [object_filter]
    title_suffix = '' if object_filter is None else f': {short_object_name(object_filter, 45)}'
    chart_times, sun_altitudes = interpolate_for_chart(hours_local, diagnostics['sun_alt'], clip=(-90, 90))
    _, moon_altitudes = interpolate_for_chart(hours_local, diagnostics['moon_alt'], clip=(-90, 90))

    figure, (altitude_ax, risk_ax) = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
    background_legend = add_solar_background(altitude_ax, chart_times, sun_altitudes)

    for obj in chart_targets:
        if obj in ('Sun', 'Moon'):
            continue
        _, object_altitudes = interpolate_for_chart(hours_local, results[obj]['alt'], clip=(-90, 90))
        altitude_ax.plot(chart_times, object_altitudes, linewidth=1.8, label=short_object_name(obj))

    altitude_ax.plot(chart_times, sun_altitudes, color='#f59e0b', linewidth=2.4, label='Sun')
    altitude_ax.plot(chart_times, moon_altitudes, color='#64748b', linewidth=2.4, label='Moon')
    add_solar_boundaries(altitude_ax, sun_limit)
    altitude_ax.axvline(current_local, color='red', linestyle='--', linewidth=1.5, label='Current time')
    altitude_ax.set_title(f"Target, Moon, and Sun altitudes{title_suffix}")
    altitude_ax.set_ylabel("Altitude (°)")
    altitude_ax.set_ylim(-95, 95)
    altitude_ax.grid(True, alpha=0.3)
    tracks_legend = altitude_ax.legend(loc='upper left', fontsize=8, ncol=2)
    altitude_ax.add_artist(tracks_legend)
    altitude_ax.legend(handles=background_legend, loc='lower left', fontsize=7, ncol=3)

    risk_ax.axhspan(0, 25, color='#86efac', alpha=0.25, label='0–25% low')
    risk_ax.axhspan(25, 50, color='#fde68a', alpha=0.25, label='25–50% noticeable')
    risk_ax.axhspan(50, 75, color='#fdba74', alpha=0.25, label='50–75% high')
    risk_ax.axhspan(75, 100, color='#fca5a5', alpha=0.25, label='75–100% critical')

    plotted_risks = 0
    for obj in chart_targets:
        obj_diagnostics = diagnostics['objects'].get(obj)
        if obj_diagnostics is None:
            continue
        _, moon_risk = interpolate_for_chart(hours_local, obj_diagnostics['moon_risk'], clip=(0, 100))
        _, moon_separation = interpolate_for_chart(hours_local, obj_diagnostics['moon_separation'], clip=(0, 180))
        max_index = int(np.argmax(moon_risk))
        max_risk = moon_risk[max_index]
        max_separation = moon_separation[max_index]
        risk_ax.plot(
            chart_times,
            moon_risk,
            linewidth=2,
            label=f"{short_object_name(obj, 24)} | max {max_risk:.0f}% | Moon sep. {max_separation:.0f}°"
        )
        risk_ax.scatter(chart_times[max_index], max_risk, marker='x', s=55, linewidths=2)
        plotted_risks += 1

    if not plotted_risks:
        risk_ax.text(
            0.5, 0.55,
            "The Moonlight penalty does not apply to the selected targets",
            transform=risk_ax.transAxes,
            ha='center',
            va='center'
        )

    risk_ax.axvline(current_local, color='red', linestyle='--', linewidth=1.5, label='Current time')
    risk_ax.set_title("Interpolated Moonlight interference risk for the selected DSOs")
    risk_ax.set_ylabel("Moonlight impact (%)")
    risk_ax.set_ylim(0, 100)
    risk_ax.grid(True, alpha=0.3)
    risk_ax.legend(loc='upper left', fontsize=8, ncol=2)
    risk_ax.xaxis.set_major_locator(mdates.HourLocator(interval=1, tz=hours_local.dt.tz))
    risk_ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M\n%d.%m', tz=hours_local.dt.tz))
    figure.tight_layout()
    plt.show()
    plt.close(figure)

def escape_ics_text(value):
    return str(value).replace('\\', '\\\\').replace(';', '\\;').replace(',', '\\,').replace('\n', '\\n')

def add_ics_event(lines, obj, start, end, ev_p, ev_t, ev_c, ev_pr, ev_w, ev_g, ics_threshold, generated_at):
    description = (
        f"Calendar threshold: {ics_threshold}%\n"
        f"Peak probability: {max(ev_p)}%\n"
        f"Temperature (avg.): {np.mean(ev_t):.1f}°C\n"
        f"Cloud cover (avg.): {np.mean(ev_c):.0f}%\n"
        f"Precipitation (max.): {max(ev_pr)}%\n"
        f"Wind (avg.): {np.mean(ev_w):.1f} m/s (gusts up to {max(ev_g):.1f} m/s)"
    )
    summary = f"Astro: {obj.split(' - ')[0]} (P~{int(np.mean(ev_p))}%)"
    uid = uuid5(NAMESPACE_URL, f"astro-planner:{obj}:{start.isoformat()}:{end.isoformat()}")
    lines.extend([
        "BEGIN:VEVENT",
        f"UID:{uid}@astro-planner",
        f"DTSTAMP:{generated_at.strftime('%Y%m%dT%H%M%SZ')}",
        f"SUMMARY:{escape_ics_text(summary)}",
        f"DTSTART:{start.strftime('%Y%m%dT%H%M%SZ')}",
        f"DTEND:{end.strftime('%Y%m%dT%H%M%SZ')}",
        f"DESCRIPTION:{escape_ics_text(description)}",
        "END:VEVENT"
    ])

def make_ics(df_weather, results, targets, ics_threshold, filename="astro.ics"):
    lines = [
        "BEGIN:VCALENDAR",
        "VERSION:2.0",
        "PRODID:-//Astro Planner//RU",
        "CALSCALE:GREGORIAN"
    ]
    generated_at = datetime.now(timezone.utc)
    
    for obj in targets:
        probs = results[obj]['p']
        in_event = False
        start = None
        ev_p, ev_t, ev_c, ev_pr, ev_w, ev_g = [], [], [], [], [], []
        
        for idx, (_, row) in enumerate(df_weather.iterrows()):
            p = probs[idx]
            t = row['time'].to_pydatetime()
            
            if p >= ics_threshold and not in_event:
                in_event, start = True, t
                ev_p, ev_t, ev_c = [p], [row['temp']], [row['cloud_cover']]
                ev_pr, ev_w, ev_g = [row['precip_prob']], [row['wind']], [row['gusts']]
                
            elif p >= ics_threshold and in_event:
                ev_p.append(p); ev_t.append(row['temp']); ev_c.append(row['cloud_cover'])
                ev_pr.append(row['precip_prob']); ev_w.append(row['wind']); ev_g.append(row['gusts'])
                
            elif p < ics_threshold and in_event:
                add_ics_event(lines, obj, start, t, ev_p, ev_t, ev_c, ev_pr, ev_w, ev_g, ics_threshold, generated_at)
                in_event = False
                
        if in_event: 
            end = df_weather.iloc[-1]['time'].to_pydatetime() + timedelta(hours=1)
            add_ics_event(lines, obj, start, end, ev_p, ev_t, ev_c, ev_pr, ev_w, ev_g, ics_threshold, generated_at)
            
    lines.append("END:VCALENDAR")
    with open(filename, "w", encoding="utf-8") as f: 
        f.write("\r\n".join(lines) + "\r\n")

# --- 6. UI AND GEOGRAPHIC FILTERING ---
_astro_planner_runtime = globals().get('_astro_planner_runtime', {
    'generation': 0,
    'running': False,
    'last_finished': 0.0,
})
_astro_planner_runtime['generation'] += 1
_astro_planner_runtime['running'] = False
planner_generation = _astro_planner_runtime['generation']

def detach_registered_callbacks():
    for kind, widget, callback in globals().get('_astro_planner_callbacks', ()):
        if kind == 'click':
            widget.on_click(callback, remove=True)
        elif kind == 'observe':
            widget.unobserve(callback, names='value')

detach_registered_callbacks()

# Compatibility cleanup for interfaces created before the callback registry existed.
try:
    btn_calc.on_click(run_pipeline, remove=True)
except NameError:
    pass

try:
    astro_planner_ui.layout.display = 'none'
except NameError:
    pass

# Remove orphaned interfaces left by notebook runs made before cleanup existed.
widgets.Widget.close_all()
plt.close('all')

lat_input = widgets.BoundedFloatText(value=55.75, min=-90, max=90, description='Latitude:')
lon_input = widgets.BoundedFloatText(value=37.61, min=-180, max=180, description='Longitude:')
sun_limit_input = widgets.BoundedFloatText(value=-18, min=-90, max=90, description='Max. Sun altitude (°):')
min_alt_input = widgets.BoundedFloatText(value=15, min=0, max=89.9, description='Min. altitude (°):')

ics_threshold_input = widgets.IntSlider(
    value=50, min=10, max=90, step=5, 
    description='Calendar threshold (%):',
    style={'description_width': 'initial'},
    layout={'width': '400px'}
)

search_box = widgets.Text(placeholder='For example: M31, NGC7000, Sun...', description='Search:')
search_results = widgets.SelectMultiple(options=[], description='Available:', layout={'height': '150px'})
selected_list = widgets.SelectMultiple(options=[], description='Selected:', layout={'height': '150px'})
btn_refresh_visible = widgets.Button(description="Refresh visible targets", button_style='primary', layout={'width':'400px'})

filtered_objects_cache = []

def update_visible_objects(*args):
    global filtered_objects_cache
    lat = lat_input.value
    m_alt = min_alt_input.value
    sys_list = list(solar_system_dict.keys())
    
    mask = (90 - (lat - dso_catalog['DecFloat']).abs()) >= m_alt
    visible_dso = dso_catalog[mask]['Name'].tolist()
    
    filtered_objects_cache = sys_list + visible_dso
    search_results.options = filtered_objects_cache[:50]
    search_box.value = ''
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Filter refreshed. Targets available at this latitude: {len(filtered_objects_cache)}")

def on_search(*args):
    q = search_box.value.lower()
    if not q: search_results.options = filtered_objects_cache[:50]
    else: search_results.options = [o for o in filtered_objects_cache if q in o.lower()][:100]

btn_refresh_visible.on_click(update_visible_objects)
search_box.observe(on_search, 'value')

btn_add = widgets.Button(description="Add ->", button_style='info')
btn_remove = widgets.Button(description="<- Remove", button_style='warning')

def add_selected_objects(*args):
    selected_list.options = list(selected_list.options) + [
        item for item in search_results.value if item not in selected_list.options
    ]

def remove_selected_objects(*args):
    selected_list.options = [
        item for item in selected_list.options if item not in selected_list.value
    ]

btn_add.on_click(add_selected_objects)
btn_remove.on_click(remove_selected_objects)

btn_calc = widgets.Button(description="CALCULATE SESSION (24 hours)", button_style='success', layout={'width':'400px'})
out = widgets.Output()

def run_pipeline_once():
    out.clear_output(wait=True)
    with out:
        plt.close('all')
        targets = list(selected_list.options)
        if not targets:
            print("Error: select targets for calculation!")
            return

        try:
            validate_inputs(lat_input.value, lon_input.value, sun_limit_input.value, min_alt_input.value)
        except ValueError as e:
            print(f"Invalid settings: {e}")
            return
            
        print("Downloading the next 24 hours of weather and calculating ephemerides (including the Moon)...")
        try:
            df = get_24h_weather(lat_input.value, lon_input.value)
        except Exception as e:
            print(f"Weather download error: {e}")
            return
            
        try:
            res = calculate_scoring(lat_input.value, lon_input.value, df, targets, sun_limit_input.value, min_alt_input.value)
            light_diagnostics = calculate_light_pollution_diagnostics(lat_input.value, lon_input.value, df, targets)
        except Exception as e:
            print(f"Ephemeris calculation error: {e}")
            return
        
        local_tz = ZoneInfo(df.attrs['display_timezone'])
        # Chart format with line wrapping
        hours_graph = df['time'].dt.tz_convert(local_tz).dt.strftime('%H:%M\n%d.%m')
        hours_local = df['time'].dt.tz_convert(local_tz)
        # Compact table format
        hours_table = df['time'].dt.tz_convert(local_tz).dt.strftime('%H:%M %d.%m')
        
        # Render charts
        chance_fig, chance_ax = plt.subplots(figsize=(14, 5))
        for obj in targets:
            chance_ax.plot(hours_graph, res[obj]['p'], marker='o', label=obj[:30]+'...' if len(obj)>30 else obj)
        chance_ax.set_title("Successful capture probability (next 24 hours)")
        chance_ax.set_ylim(-5, 105)
        chance_ax.grid(True, alpha=0.3)
        chance_ax.legend()
        chance_ax.tick_params(axis='x', rotation=45)
        chance_fig.tight_layout()
        plt.show()
        plt.close(chance_fig)

        # Target altitude and interpolated Moonlight interference risk
        current_local = datetime.now(timezone.utc).astimezone(local_tz)
        if len(targets) <= 5:
            plot_light_pollution_chart(
                hours_local, res, light_diagnostics, targets,
                sun_limit_input.value, current_local
            )
        else:
            for obj in targets:
                plot_light_pollution_chart(
                    hours_local, res, light_diagnostics, targets,
                    sun_limit_input.value, current_local,
                    object_filter=obj
                )
        
        # Build the extended review matrix
        print(f"\nHourly summary matrix (calendar threshold: {ics_threshold_input.value}%):")
        matrix = {
            'Time (local)': hours_table,
            'Cloud cover (%)': [f"{c:.0f}%" for c in df['cloud_cover']],
            'Temp. (°C)': [f"{t:.1f}" for t in df['temp']],
            'Humidity (%)': [f"{h:.0f}%" for h in df['humidity']],
            'Precipitation': [f"{p:.0f}%" for p in df['precip_prob']],
            'Wind (m/s)': [f"{w:.1f} (up to {g:.1f})" for w, g in zip(df['wind'], df['gusts'])]
        }
        
        # Calculate Moon phase and altitude for each hour
        moon_data = []
        observer = earth + Topos(latitude_degrees=lat_input.value, longitude_degrees=lon_input.value)
        for _, row in df.iterrows():
            t_utc = ts.from_datetime(row['time'])
            moon_alt, _, _ = observer.at(t_utc).observe(moon).apparent().altaz()
            if moon_alt.degrees > 0:
                phase = fraction_illuminated(planets, 'moon', t_utc) * 100
                moon_data.append(f"{phase:.0f}% (Alt: {moon_alt.degrees:.0f}°)")
            else:
                moon_data.append("Below horizon")
        matrix['Moon'] = moon_data

        # Add target scores
        for obj in targets:
            matrix[f"{obj} | Altitude"] = [f"{alt:.0f}°" for alt in res[obj]['alt']]
            matrix[f"{obj} | Probability"] = [
                f"{p}% ({reason})"
                for p, reason in zip(res[obj]['p'], res[obj]['r'])
            ]
            
        display(pd.DataFrame(matrix).style.set_properties(**{'text-align': 'left'}))

        make_ics(df, res, targets, ics_threshold_input.value)
        print(f"\nastro.ics calendar generated successfully for windows >= {ics_threshold_input.value}%.")

def run_pipeline(*args, _generation=planner_generation):
    runtime = _astro_planner_runtime
    if _generation != runtime['generation']:
        return

    now = time.monotonic()
    if runtime['running'] or now - runtime['last_finished'] < 1.0:
        return

    runtime['running'] = True
    btn_calc.disabled = True
    try:
        run_pipeline_once()
    finally:
        runtime['running'] = False
        runtime['last_finished'] = time.monotonic()
        btn_calc.disabled = False

# Keep exactly one dispatcher on the active button.
btn_calc._click_handlers.callbacks.clear()
btn_calc.on_click(run_pipeline)

_astro_planner_callbacks = [
    ('click', btn_refresh_visible, update_visible_objects),
    ('observe', search_box, on_search),
    ('click', btn_add, add_selected_objects),
    ('click', btn_remove, remove_selected_objects),
    ('click', btn_calc, run_pipeline),
]

# Build the interface
update_visible_objects()
clear_output()
print(f"=== ASTRO PLANNER (V2) READY | BUILD {APP_BUILD} ===")
astro_planner_ui = widgets.VBox([
    widgets.HTML("<b>1. Coordinates and equipment limits</b>"), 
    widgets.HBox([lat_input, lon_input]),
    widgets.HBox([sun_limit_input, min_alt_input]),
    ics_threshold_input,
    btn_refresh_visible,
    widgets.HTML("<hr><b>2. Target selection (potentially visible at the selected latitude)</b>"), 
    widgets.HBox([
        widgets.VBox([search_box, search_results]),
        widgets.VBox([btn_add, btn_remove], layout=widgets.Layout(justify_content='center')),
        selected_list
    ]),
    widgets.HTML("<hr>"), btn_calc, out
])
display(astro_planner_ui)


=== ASTRO PLANNER (V2) READY | BUILD 2026-06-01-lightpollution-v1 ===


In [2]:
"""
Sector-based astrophotography recommendations.

Run this cell after the first cell of astro_planner_lightpollution.ipynb.
It deliberately uses its own namespace and never modifies the first UI.
"""

import os
import time
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
import ipywidgets as widgets
from IPython.display import clear_output, display
from skyfield.api import Topos
from skyfield.almanac import fraction_illuminated


_sector_planner_required_globals = [
    "earth",
    "sun",
    "moon",
    "planets",
    "solar_system_dict",
    "ts",
    "get_24h_weather",
    "calculate_scoring",
    "calculate_light_pollution_diagnostics",
    "plot_light_pollution_chart",
    "lat_input",
    "lon_input",
    "min_alt_input",
    "sun_limit_input",
    "ics_threshold_input",
]
_sector_planner_missing = [
    name for name in _sector_planner_required_globals if name not in globals()
]
if _sector_planner_missing:
    raise RuntimeError(
        "Run the first planner cell before this cell. "
        "Missing: " + ", ".join(_sector_planner_missing)
    )


def _sector_planner_close_widget(widget):
    if widget is None:
        return
    for child in getattr(widget, "children", ()):
        _sector_planner_close_widget(child)
    try:
        widget.close()
    except Exception:
        pass


def _sector_planner_detach_previous():
    for widget, callback in globals().get("_sector_planner_callbacks", []):
        try:
            widget.on_click(callback, remove=True)
        except Exception:
            pass
    for widget, callback, trait in globals().get("_sector_planner_observers", []):
        try:
            widget.unobserve(callback, names=trait)
        except Exception:
            pass
    for link in globals().get("_sector_planner_links", []):
        try:
            link.unlink()
        except Exception:
            pass
    _sector_planner_close_widget(globals().get("_sector_planner_ui"))


_sector_planner_detach_previous()
_sector_planner_generation = globals().get("_sector_planner_generation", 0) + 1
_sector_planner_callbacks = []
_sector_planner_observers = []
_sector_planner_links = []
_sector_planner_snapshot = None
_sector_planner_selected = []


_sector_planner_type_map = {
    "*": "Star",
    "**": "Double star",
    "*Ass": "Stellar association",
    "OCl": "Open cluster",
    "GCl": "Globular cluster",
    "Cl+N": "Cluster with nebulosity",
    "G": "Galaxy",
    "GPair": "Galaxy pair",
    "GTrpl": "Galaxy triplet",
    "GGroup": "Galaxy group",
    "PN": "Planetary nebula",
    "HII": "HII region",
    "Neb": "Nebula",
    "EmN": "Emission nebula",
    "RfN": "Reflection nebula",
    "SNR": "Supernova remnant",
    "Nova": "Nova",
    "NonEx": "Nonexistent",
    "Other": "Other",
}
_sector_planner_first_cell_type_map = {
    "G": "Galaxy",
    "PN": "Planetary nebula",
    "OC": "Open cluster",
    "GC": "Globular cluster",
    "Nb": "Nebula",
    "C+N": "Cluster + nebula",
    "Ast": "Asterism",
    "Kt": "Knot",
    "*": "Star",
    "D*": "Double star",
    "***": "Triple star",
    "SC": "Star cloud",
    "HII": "HII region",
}
_sector_planner_dso_types = {
    "G",
    "GPair",
    "GTrpl",
    "GGroup",
    "OCl",
    "GCl",
    "PN",
    "Neb",
    "HII",
    "Cl+N",
    "RfN",
    "SNR",
    "EmN",
    "*Ass",
}


def _sector_planner_parse_ra(value):
    try:
        parts = str(value).strip().split(":")
        return (float(parts[0]) + float(parts[1]) / 60 + float(parts[2]) / 3600) * 15
    except Exception:
        return np.nan


def _sector_planner_parse_dec(value):
    try:
        text = str(value).strip()
        sign = -1 if text.startswith("-") else 1
        parts = text.lstrip("+-").split(":")
        return sign * (
            float(parts[0]) + float(parts[1]) / 60 + float(parts[2]) / 3600
        )
    except Exception:
        return np.nan


def _sector_planner_numeric(frame, column):
    if column not in frame:
        return pd.Series(np.nan, index=frame.index, dtype=float)
    return pd.to_numeric(frame[column], errors="coerce")


def _sector_planner_inverse_percentile(values, default=0.35):
    numeric = pd.to_numeric(values, errors="coerce")
    return (1.0 - numeric.rank(pct=True)).fillna(default).clip(0, 1)


def _sector_planner_build_catalog():
    cached = globals().get("_sector_planner_extended_catalog")
    if cached is not None:
        return cached

    catalog_url = (
        "https://raw.githubusercontent.com/mattiaverga/OpenNGC/master/"
        "database_files/NGC.csv"
    )
    catalog_path = "NGC_full_decoded.csv" if os.path.exists("NGC_full_decoded.csv") else catalog_url
    frame = pd.read_csv(catalog_path, sep=";", low_memory=False)
    frame["RaFloat"] = frame["RA"].map(_sector_planner_parse_ra)
    frame["DecFloat"] = frame["Dec"].map(_sector_planner_parse_dec)
    frame = frame[np.isfinite(frame["RaFloat"]) & np.isfinite(frame["DecFloat"])].copy()
    frame["CatalogName"] = frame["Name"].fillna("").astype(str).str.strip()
    messier = frame.get("M", pd.Series("", index=frame.index)).fillna("").astype(str).str.strip()
    frame["CommonName"] = (
        frame.get("Common names", pd.Series("", index=frame.index))
        .fillna("")
        .astype(str)
        .str.strip()
    )

    compatible_type = (
        frame["Type"].fillna("").astype(str).str.strip().map(_sector_planner_first_cell_type_map)
    ).fillna("DSO")
    frame["Name"] = frame["CatalogName"] + " - " + compatible_type
    messier_number = pd.to_numeric(frame.get("M"), errors="coerce")
    has_messier = messier_number.notna()
    frame.loc[has_messier, "Name"] = (
        "M"
        + messier_number[has_messier].astype(int).astype(str)
        + " ("
        + frame.loc[has_messier, "CatalogName"]
        + ") - "
        + compatible_type[has_messier]
    )
    frame["V-Mag"] = _sector_planner_numeric(frame, "V-Mag")
    frame["B-Mag"] = _sector_planner_numeric(frame, "B-Mag")
    frame["SurfBr"] = _sector_planner_numeric(frame, "SurfBr")
    frame["MajAx"] = _sector_planner_numeric(frame, "MajAx")
    frame["MinAx"] = _sector_planner_numeric(frame, "MinAx")
    frame["BrightnessMag"] = frame["V-Mag"].combine_first(frame["B-Mag"])
    mag_score = _sector_planner_inverse_percentile(frame["BrightnessMag"])
    surface_score = _sector_planner_inverse_percentile(frame["SurfBr"])
    frame["BrightnessScore"] = (0.65 * mag_score + 0.35 * surface_score).clip(0, 1)
    size_rank = np.log1p(frame["MajAx"].clip(lower=0)).rank(pct=True)
    frame["SizeScore"] = size_rank.fillna(0.35).clip(0, 1)
    frame["TypeLabel"] = frame["Type"].map(_sector_planner_type_map).fillna(frame["Type"])
    frame = frame.drop_duplicates("Name").reset_index(drop=True)
    globals()["_sector_planner_extended_catalog"] = frame
    return frame


def _sector_planner_circular_delta(angle, center):
    return (np.asarray(angle) - float(center) + 180.0) % 360.0 - 180.0


def _sector_planner_sky_geometry(ra_deg, dec_deg, lat, lon, skyfield_times):
    ra = np.radians(np.asarray(ra_deg, dtype=float))
    dec = np.radians(np.asarray(dec_deg, dtype=float))
    lat_rad = np.radians(float(lat))
    lst = np.radians((np.asarray(skyfield_times.gast) * 15.0 + float(lon)) % 360.0)
    hour_angle = lst[:, None] - ra[None, :]
    sin_alt = (
        np.sin(lat_rad) * np.sin(dec)[None, :]
        + np.cos(lat_rad) * np.cos(dec)[None, :] * np.cos(hour_angle)
    )
    alt = np.degrees(np.arcsin(np.clip(sin_alt, -1, 1)))
    y = -np.sin(hour_angle) * np.cos(dec)[None, :]
    x = (
        np.sin(dec)[None, :] * np.cos(lat_rad)
        - np.cos(dec)[None, :] * np.sin(lat_rad) * np.cos(hour_angle)
    )
    az = np.degrees(np.mod(np.arctan2(y, x), 2 * np.pi))
    return alt, az


def _sector_planner_moon_risk(moon_alt, moon_phase, moon_ra, moon_dec, ra, dec):
    moon_ra_rad = np.radians(np.asarray(moon_ra, dtype=float))[:, None]
    moon_dec_rad = np.radians(np.asarray(moon_dec, dtype=float))[:, None]
    ra_rad = np.radians(np.asarray(ra, dtype=float))[None, :]
    dec_rad = np.radians(np.asarray(dec, dtype=float))[None, :]
    cos_sep = (
        np.sin(moon_dec_rad) * np.sin(dec_rad)
        + np.cos(moon_dec_rad) * np.cos(dec_rad) * np.cos(moon_ra_rad - ra_rad)
    )
    separation = np.degrees(np.arccos(np.clip(cos_sep, -1, 1)))
    risk = (
        np.asarray(moon_phase, dtype=float)[:, None]
        * np.maximum(np.asarray(moon_alt, dtype=float)[:, None], 0)
        / 90.0
        * (1.0 - np.minimum(separation, 180.0) / 180.0)
        * 1.5
        * 100.0
    )
    return np.clip(risk, 0, 100), separation


def _sector_planner_runs(mask):
    result = []
    start = None
    for index, value in enumerate(np.asarray(mask, dtype=bool)):
        if value and start is None:
            start = index
        if start is not None and (not value or index == len(mask) - 1):
            stop = index if value and index == len(mask) - 1 else index - 1
            result.append((start, stop))
            start = None
    return result


def _sector_planner_best_window(probability, inside, threshold, local_times):
    probability = np.asarray(probability, dtype=float)
    strong_runs = _sector_planner_runs(probability >= float(threshold))
    weak = not strong_runs
    candidates = strong_runs or _sector_planner_runs(np.asarray(inside, dtype=bool))
    if not candidates:
        return {
            "Weak": True,
            "Start": None,
            "Stop": None,
            "Hours": 0,
            "Mean": 0.0,
            "Max": 0.0,
            "Score": 0.0,
            "Label": "no window",
        }

    def run_key(run):
        start, stop = run
        values = probability[start : stop + 1]
        return (stop - start + 1, float(np.mean(values)), float(np.max(values)))

    start, stop = max(candidates, key=run_key)
    values = probability[start : stop + 1]
    hours = stop - start + 1
    quality = 0.65 * float(np.mean(values)) / 100.0 + 0.35 * min(hours, 4) / 4.0
    if weak:
        quality *= 0.75
    prefix = "weak window: " if weak else ""
    label = (
        prefix
        + pd.Timestamp(local_times[start]).strftime("%d.%m %H:%M")
        + "–"
        + pd.Timestamp(local_times[stop]).strftime("%H:%M")
    )
    return {
        "Weak": weak,
        "Start": start,
        "Stop": stop,
        "Hours": hours,
        "Mean": float(np.mean(values)),
        "Max": float(np.max(values)),
        "Score": float(np.clip(quality, 0, 1)),
        "Label": label,
    }


def _sector_planner_best_windows_batch(probability, inside, threshold, local_times):
    probability = np.asarray(probability, dtype=float)
    inside = np.asarray(inside, dtype=bool)
    strong = probability >= float(threshold)
    weak = ~np.any(strong, axis=0)
    active = np.where(weak[None, :], inside, strong)
    count = active.shape[1]
    current_start = np.zeros(count, dtype=int)
    current_length = np.zeros(count, dtype=int)
    current_sum = np.zeros(count, dtype=float)
    current_max = np.zeros(count, dtype=float)
    best_start = np.full(count, -1, dtype=int)
    best_stop = np.full(count, -1, dtype=int)
    best_length = np.zeros(count, dtype=int)
    best_sum = np.zeros(count, dtype=float)
    best_max = np.zeros(count, dtype=float)

    for hour in range(active.shape[0]):
        is_active = active[hour]
        begins = is_active & (current_length == 0)
        current_start = np.where(begins, hour, current_start)
        current_length = np.where(is_active, current_length + 1, 0)
        current_sum = np.where(is_active, current_sum + probability[hour], 0)
        current_max = np.where(is_active, np.maximum(current_max, probability[hour]), 0)
        current_mean = np.divide(
            current_sum,
            current_length,
            out=np.zeros(count, dtype=float),
            where=current_length > 0,
        )
        best_mean = np.divide(
            best_sum,
            best_length,
            out=np.zeros(count, dtype=float),
            where=best_length > 0,
        )
        better = is_active & (
            (current_length > best_length)
            | ((current_length == best_length) & (current_mean > best_mean))
            | (
                (current_length == best_length)
                & np.isclose(current_mean, best_mean)
                & (current_max > best_max)
            )
        )
        best_start = np.where(better, current_start, best_start)
        best_stop = np.where(better, hour, best_stop)
        best_length = np.where(better, current_length, best_length)
        best_sum = np.where(better, current_sum, best_sum)
        best_max = np.where(better, current_max, best_max)

    best_mean = np.divide(
        best_sum,
        best_length,
        out=np.zeros(count, dtype=float),
        where=best_length > 0,
    )
    score = 0.65 * best_mean / 100.0 + 0.35 * np.minimum(best_length, 4) / 4.0
    score = np.where(weak, score * 0.75, score)
    start_labels = local_times.dt.strftime("%d.%m %H:%M").to_numpy()
    stop_labels = local_times.dt.strftime("%H:%M").to_numpy()
    labels = []
    for start, stop, is_weak in zip(best_start, best_stop, weak):
        if start < 0:
            labels.append("no window")
            continue
        prefix = "weak window: " if is_weak else ""
        labels.append(prefix + start_labels[start] + "–" + stop_labels[stop])
    return {
        "Weak": weak,
        "Start": best_start,
        "Stop": best_stop,
        "Hours": best_length,
        "Mean": best_mean,
        "Max": best_max,
        "Score": np.clip(score, 0, 1),
        "Label": np.asarray(labels, dtype=object),
    }


def _sector_planner_center_quality(alt, az, inside, direction, low, high, half_width):
    alt = np.asarray(alt, dtype=float)
    az = np.asarray(az, dtype=float)
    inside = np.asarray(inside, dtype=bool)
    alt_half = max((float(high) - float(low)) / 2.0, 1e-6)
    az_half = max(float(half_width), 1e-6)
    alt_distance = np.abs(alt - (float(low) + float(high)) / 2.0) / alt_half
    az_distance = np.abs(_sector_planner_circular_delta(az, direction)) / az_half
    quality = 1.0 - np.clip(np.sqrt((alt_distance**2 + az_distance**2) / 2.0), 0, 1)
    return float(np.mean(quality[inside])) if np.any(inside) else 0.0


def _sector_planner_center_qualities(alt, az, inside, direction, low, high, half_width):
    alt = np.asarray(alt, dtype=float)
    az = np.asarray(az, dtype=float)
    inside = np.asarray(inside, dtype=bool)
    alt_half = max((float(high) - float(low)) / 2.0, 1e-6)
    az_half = max(float(half_width), 1e-6)
    alt_distance = np.abs(alt - (float(low) + float(high)) / 2.0) / alt_half
    az_distance = np.abs(_sector_planner_circular_delta(az, direction)) / az_half
    quality = 1.0 - np.clip(np.sqrt((alt_distance**2 + az_distance**2) / 2.0), 0, 1)
    return np.divide(
        np.sum(np.where(inside, quality, 0.0), axis=0),
        np.sum(inside, axis=0),
        out=np.zeros(inside.shape[1], dtype=float),
        where=np.sum(inside, axis=0) > 0,
    )


def _sector_planner_weather_array(weather, column, default):
    if column not in weather:
        return np.full(len(weather), default, dtype=float)
    return pd.to_numeric(weather[column], errors="coerce").fillna(default).to_numpy(float)


def _sector_planner_dso_probability(
    alt, inside, sun_alt, moon_risk, weather, min_alt, sun_limit
):
    alt_factor = np.clip((alt - float(min_alt)) / max(90.0 - float(min_alt), 1e-6), 0, 1)
    cloud = _sector_planner_weather_array(weather, "cloud_cover", 100)
    precipitation = _sector_planner_weather_array(weather, "precip_prob", 100)
    gust = _sector_planner_weather_array(weather, "gusts", 50)
    weather_factor = (
        np.clip(1.0 - cloud / 100.0, 0, 1)
        * np.clip(1.0 - precipitation / 100.0, 0, 1)
        * np.clip(1.0 - np.maximum(0, gust - 5) / 10.0, 0, 1)
    )[:, None]
    probability = (
        100.0
        * (0.6 + 0.4 * alt_factor)
        * weather_factor
        * (1.0 - np.asarray(moon_risk, dtype=float) / 100.0)
    )
    allowed = (
        np.asarray(inside, dtype=bool)
        & (alt >= float(min_alt))
        & (np.asarray(sun_alt, dtype=float)[:, None] <= float(sun_limit))
    )
    return np.where(allowed, np.clip(probability, 0, 100), 0.0)


def _sector_planner_reason(probability, inside, altitude, sun_altitude, moon_risk, settings):
    if not inside:
        return "Outside sector"
    if altitude < settings["min_alt"]:
        return "Below minimum altitude"
    if sun_altitude > settings["sun_limit"]:
        return "Sky too bright"
    if moon_risk >= 50:
        return "High Moonlight interference"
    if probability <= 0:
        return "Poor weather"
    return "Suitable for capture"


def _sector_planner_solar_probability(name, altitude, sun_altitude, weather, min_alt):
    altitude = np.asarray(altitude, dtype=float)
    sun_altitude = np.asarray(sun_altitude, dtype=float)
    cloud = _sector_planner_weather_array(weather, "cloud_cover", 100)
    precipitation = _sector_planner_weather_array(weather, "precip_prob", 100)
    gust = _sector_planner_weather_array(weather, "gusts", 50)
    alt_factor = np.clip((altitude - float(min_alt)) / max(90 - float(min_alt), 1e-6), 0, 1)
    probability = (
        np.clip(1.0 - cloud / 100.0, 0, 1)
        * (0.6 + 0.4 * alt_factor)
        * np.clip(1.0 - precipitation / 100.0, 0, 1)
        * np.clip(1.0 - np.maximum(0, gust - 5) / 10.0, 0, 1)
        * 100
    )
    allowed = (altitude >= float(min_alt)) & (sun_altitude <= 0)
    probability = np.where(allowed, probability, 0).astype(int).astype(float)
    reasons = []
    for p, alt, solar_alt, clouds, precip, wind_gust in zip(
        probability, altitude, sun_altitude, cloud, precipitation, gust
    ):
        if alt < float(min_alt):
            reasons.append(f"Below {min_alt}°")
        elif solar_alt > 0:
            reasons.append("Sky too bright")
        elif p >= 30:
            reasons.append("OK")
        elif clouds > 70:
            reasons.append("Cloudy")
        elif precip > 20:
            reasons.append("Precipitation")
        elif wind_gust > 8:
            reasons.append("Wind")
        else:
            reasons.append("Low altitude / poor conditions")
    return probability, reasons


def _sector_planner_build_snapshot(settings, weather):
    started = time.perf_counter()
    catalog = _sector_planner_build_catalog()
    shooting = catalog[catalog["Type"].isin(_sector_planner_dso_types)].copy()
    shooting_indices = shooting.index.to_numpy()
    utc_times = pd.to_datetime(weather["time"], utc=True)
    local_zone = ZoneInfo(weather.attrs.get("display_timezone", "UTC"))
    local_times = utc_times.dt.tz_convert(local_zone)
    skyfield_times = ts.from_datetimes([item.to_pydatetime() for item in utc_times])
    observer = earth + Topos(
        latitude_degrees=settings["lat"], longitude_degrees=settings["lon"]
    )
    sun_alt = (
        observer.at(skyfield_times).observe(sun).apparent().altaz()[0].degrees
    )
    moon_apparent = observer.at(skyfield_times).observe(moon).apparent()
    moon_alt = moon_apparent.altaz()[0].degrees
    moon_ra, moon_dec, _ = moon_apparent.radec()
    moon_ra = moon_ra.hours * 15.0
    moon_dec = moon_dec.degrees
    moon_phase = np.asarray(fraction_illuminated(planets, "moon", skyfield_times), dtype=float)

    alt, az = _sector_planner_sky_geometry(
        shooting["RaFloat"], shooting["DecFloat"], settings["lat"], settings["lon"], skyfield_times
    )
    inside = (
        (alt >= settings["alt_low"])
        & (alt <= settings["alt_high"])
        & (
            np.abs(_sector_planner_circular_delta(az, settings["direction"]))
            <= settings["half_width"]
        )
    )
    moon_risk, moon_separation = _sector_planner_moon_risk(
        moon_alt,
        moon_phase,
        moon_ra,
        moon_dec,
        shooting["RaFloat"],
        shooting["DecFloat"],
    )
    probability = _sector_planner_dso_probability(
        alt,
        inside,
        sun_alt,
        moon_risk,
        weather,
        settings["min_alt"],
        settings["sun_limit"],
    )
    candidate_indices = np.flatnonzero(np.any(inside, axis=0))
    dso_windows = _sector_planner_best_windows_batch(
        probability[:, candidate_indices],
        inside[:, candidate_indices],
        settings["threshold"],
        local_times,
    )
    dso_centers = _sector_planner_center_qualities(
        alt[:, candidate_indices],
        az[:, candidate_indices],
        inside[:, candidate_indices],
        settings["direction"],
        settings["alt_low"],
        settings["alt_high"],
        settings["half_width"],
    )
    shooting_names = shooting["Name"].to_numpy()
    shooting_types = shooting["TypeLabel"].to_numpy()
    shooting_brightness = shooting["BrightnessScore"].to_numpy(float)
    shooting_sizes = shooting["SizeScore"].to_numpy(float)
    shooting_magnitudes = shooting["BrightnessMag"].to_numpy(float)
    shooting_major_axes = shooting["MajAx"].to_numpy(float)
    records = []
    for candidate_index, matrix_index in enumerate(candidate_indices):
        raw_score = 100.0 * (
            0.40 * dso_windows["Score"][candidate_index]
            + 0.25 * dso_centers[candidate_index]
            + 0.25 * shooting_brightness[matrix_index]
            + 0.10 * shooting_sizes[matrix_index]
        )
        records.append(
            {
                "Name": shooting_names[matrix_index],
                "Family": "DSO",
                "Type": shooting_types[matrix_index],
                "RawScore": raw_score,
                "Window": dso_windows["Label"][candidate_index],
                "WindowHours": int(dso_windows["Hours"][candidate_index]),
                "WeakWindow": bool(dso_windows["Weak"][candidate_index]),
                "MeanProbability": float(dso_windows["Mean"][candidate_index]),
                "MaxProbability": float(np.max(probability[:, matrix_index])),
                "SectorHours": int(np.count_nonzero(inside[:, matrix_index])),
                "Magnitude": shooting_magnitudes[matrix_index],
                "Size": shooting_major_axes[matrix_index],
                "MatrixIndex": int(matrix_index),
            }
        )

    solar_cache = {}
    for name, body in solar_system_dict.items():
        if body is sun or name == "Sun":
            continue
        apparent = observer.at(skyfield_times).observe(body).apparent()
        body_alt, body_az, _ = apparent.altaz()
        body_alt = np.asarray(body_alt.degrees, dtype=float)
        body_az = np.asarray(body_az.degrees, dtype=float)
        body_inside = (
            (body_alt >= settings["alt_low"])
            & (body_alt <= settings["alt_high"])
            & (
                np.abs(_sector_planner_circular_delta(body_az, settings["direction"]))
                <= settings["half_width"]
            )
        )
        base_probability, base_reasons = _sector_planner_solar_probability(
            name,
            body_alt,
            sun_alt,
            weather,
            settings["min_alt"],
        )
        body_probability = np.where(body_inside, base_probability, 0.0)
        body_reasons = [
            reason if in_sector else "Outside sector"
            for reason, in_sector in zip(base_reasons, body_inside)
        ]
        solar_cache[name] = {
            "alt": body_alt,
            "az": body_az,
            "inside": body_inside,
            "probability": body_probability,
            "reasons": body_reasons,
        }
        if not np.any(body_inside):
            continue
        window = _sector_planner_best_window(
            body_probability, body_inside, settings["threshold"], local_times
        )
        center = _sector_planner_center_quality(
            body_alt,
            body_az,
            body_inside,
            settings["direction"],
            settings["alt_low"],
            settings["alt_high"],
            settings["half_width"],
        )
        records.append(
            {
                "Name": name,
                "Family": "Solar System",
                "Type": "Moon" if name == "Moon" else "Planet",
                "RawScore": 100.0 * (0.65 * window["Score"] + 0.35 * center),
                "Window": window["Label"],
                "WindowHours": window["Hours"],
                "WeakWindow": window["Weak"],
                "MeanProbability": window["Mean"],
                "MaxProbability": float(np.max(body_probability)),
                "SectorHours": int(np.count_nonzero(body_inside)),
                "Magnitude": np.nan,
                "Size": np.nan,
                "MatrixIndex": np.nan,
            }
        )

    records = pd.DataFrame(records)
    if not records.empty:
        records["NormalizedScore"] = records.groupby("Family")["RawScore"].rank(pct=True) * 100
    snapshot = {
        "settings": settings,
        "weather": weather,
        "utc_times": utc_times,
        "local_times": local_times,
        "skyfield_times": skyfield_times,
        "sun_alt": np.asarray(sun_alt, dtype=float),
        "moon_alt": np.asarray(moon_alt, dtype=float),
        "moon_phase": moon_phase,
        "shooting": shooting.reset_index(drop=True),
        "shooting_indices": shooting_indices,
        "alt": alt,
        "az": az,
        "inside": inside,
        "probability": probability,
        "moon_risk": moon_risk,
        "moon_separation": moon_separation,
        "solar": solar_cache,
        "records": records,
        "elapsed": time.perf_counter() - started,
    }
    return snapshot


def _sector_planner_records_for_mode(snapshot):
    if snapshot is None or snapshot["records"].empty:
        return pd.DataFrame()
    records = snapshot["records"].copy()
    mode = _sector_target_mode.value
    if mode == "DSO":
        records = records[records["Family"] == "DSO"].copy()
        records["Score"] = records["RawScore"]
    elif mode == "Planets + Moon":
        records = records[records["Family"] == "Solar System"].copy()
        records["Score"] = records["RawScore"]
    else:
        records["Score"] = records["NormalizedScore"]
    return records.sort_values(
        ["Score", "MaxProbability", "SectorHours"], ascending=False
    ).reset_index(drop=True)


def _sector_planner_top3(snapshot):
    return _sector_planner_records_for_mode(snapshot).head(3).copy()


def _sector_planner_label(row):
    family = "DSO" if row["Family"] == "DSO" else "SS"
    return f'{row["Name"]} [{family}; {row["Score"]:.1f}]'


def _sector_planner_update_available(*_):
    snapshot = globals().get("_sector_planner_snapshot")
    records = _sector_planner_records_for_mode(snapshot)
    query = _sector_search.value.strip().lower()
    if query:
        records = records[
            records["Name"].str.lower().str.contains(query, regex=False)
            | records["Type"].str.lower().str.contains(query, regex=False)
        ]
    chosen = set(_sector_planner_selected)
    _sector_available.options = [
        (_sector_planner_label(row), row["Name"])
        for _, row in records.iterrows()
        if row["Name"] not in chosen
    ]
    _sector_chosen.options = [(name, name) for name in _sector_planner_selected]


def _sector_planner_render_top3(snapshot):
    top = _sector_planner_top3(snapshot)
    with _sector_top3_out:
        clear_output(wait=True)
        if top.empty:
            print("No targets for this mode were found in the selected sector.")
            return
        table = top[
            [
                "Name",
                "Type",
                "Score",
                "Window",
                "MeanProbability",
                "MaxProbability",
                "SectorHours",
                "Magnitude",
                "Size",
            ]
        ].copy()
        table.columns = [
            "Target",
            "Type",
            "Score",
            "Best window",
            "Average probability, %",
            "Max. probability, %",
            "Hours in sector",
            "Magnitude",
            "Size, arcmin",
        ]
        display(table.round(1))


def _sector_planner_target_arrays(snapshot, name):
    record = snapshot["records"][snapshot["records"]["Name"] == name]
    if record.empty:
        return None
    row = record.iloc[0]
    if row["Family"] == "DSO":
        index = int(row["MatrixIndex"])
        probability = snapshot["probability"][:, index]
        inside = snapshot["inside"][:, index]
        altitude = snapshot["alt"][:, index]
        azimuth = snapshot["az"][:, index]
        moon_risk = snapshot["moon_risk"][:, index]
        reasons = [
            _sector_planner_reason(p, s, a, sun_a, risk, snapshot["settings"])
            for p, s, a, sun_a, risk in zip(
                probability, inside, altitude, snapshot["sun_alt"], moon_risk
            )
        ]
    else:
        cached = snapshot["solar"][name]
        probability = cached["probability"]
        inside = cached["inside"]
        altitude = cached["alt"]
        azimuth = cached["az"]
        moon_risk = np.zeros(len(snapshot["weather"]))
        reasons = cached["reasons"]
    return {
        "probability": np.asarray(probability, dtype=float),
        "inside": np.asarray(inside, dtype=bool),
        "altitude": np.asarray(altitude, dtype=float),
        "azimuth": np.asarray(azimuth, dtype=float),
        "moon_risk": np.asarray(moon_risk, dtype=float),
        "reasons": reasons,
    }


def _sector_planner_plot_light_pollution_chart(
    hours_local,
    probability_results,
    results,
    diagnostics,
    targets,
    sun_limit,
    current_local,
    settings,
    threshold,
    object_filter=None,
):
    chart_targets = targets if object_filter is None else [object_filter]
    title_suffix = "" if object_filter is None else f": {short_object_name(object_filter, 45)}"
    chart_times, sun_altitudes = interpolate_for_chart(
        hours_local, diagnostics["sun_alt"], clip=(-90, 90)
    )
    _, moon_altitudes = interpolate_for_chart(
        hours_local, diagnostics["moon_alt"], clip=(-90, 90)
    )

    figure, (probability_ax, altitude_ax, risk_ax) = plt.subplots(
        3,
        1,
        figsize=(14, 12),
        sharex=True,
        gridspec_kw={"height_ratios": [1, 1.45, 1]},
    )
    for obj in chart_targets:
        probability_ax.plot(
            hours_local,
            probability_results[obj],
            marker="o",
            linewidth=1.8,
            label=short_object_name(obj),
        )
    probability_ax.axhline(
        threshold,
        color="tab:red",
        linestyle="--",
        linewidth=1,
        label=f"Calendar threshold {threshold:g}%",
    )
    probability_ax.set_title(f"Capture probability within the sector{title_suffix}")
    probability_ax.set_ylabel("Probability, %")
    probability_ax.set_ylim(-2, 102)
    probability_ax.grid(True, alpha=0.3)
    probability_ax.legend(loc="upper left", fontsize=8, ncol=2)

    background_legend = add_solar_background(altitude_ax, chart_times, sun_altitudes)
    sector_low = settings["alt_low"]
    sector_high = settings["alt_high"]
    sector_color = "#64748b"
    altitude_ax.axhspan(
        sector_low,
        sector_high,
        color=sector_color,
        alpha=0.10,
        label=f"Sector altitude frame {sector_low:g}–{sector_high:g}°",
    )
    altitude_ax.axhline(
        sector_low,
        color=sector_color,
        linestyle="--",
        linewidth=1.2,
        label=f"Lower sector altitude {sector_low:g}°",
    )
    altitude_ax.axhline(
        sector_high,
        color=sector_color,
        linestyle="--",
        linewidth=1.2,
        label=f"Upper sector altitude {sector_high:g}°",
    )
    altitude_ax.text(
        chart_times[-1],
        sector_low,
        f" {sector_low:g}°",
        color=sector_color,
        fontsize=9,
        ha="right",
        va="bottom",
    )
    altitude_ax.text(
        chart_times[-1],
        sector_high,
        f" {sector_high:g}°",
        color=sector_color,
        fontsize=9,
        ha="right",
        va="bottom",
    )

    for obj in chart_targets:
        if obj in ("Sun", "Moon"):
            continue
        _, object_altitudes = interpolate_for_chart(
            hours_local, results[obj]["alt"], clip=(-90, 90)
        )
        altitude_ax.plot(
            chart_times, object_altitudes, linewidth=1.8, label=short_object_name(obj)
        )

    altitude_ax.plot(
        chart_times,
        sun_altitudes,
        color="#f59e0b",
        linewidth=2.4,
        label="Sun",
    )
    altitude_ax.plot(
        chart_times,
        moon_altitudes,
        color="#64748b",
        linewidth=2.4,
        label="Moon",
    )
    add_solar_boundaries(altitude_ax, sun_limit)
    altitude_ax.axvline(
        current_local,
        color="red",
        linestyle="--",
        linewidth=1.5,
        label="Current time",
    )
    altitude_ax.set_title(
        f"Target, Moon, and Sun altitudes{title_suffix} | "
        f"sector azimuth {settings['direction']:g}° ± {settings['half_width']:g}°"
    )
    altitude_ax.set_ylabel("Altitude (°)")
    altitude_ax.set_ylim(-95, 95)
    altitude_ax.grid(True, alpha=0.3)
    tracks_legend = altitude_ax.legend(loc="upper left", fontsize=8, ncol=2)
    altitude_ax.add_artist(tracks_legend)
    altitude_ax.legend(handles=background_legend, loc="lower left", fontsize=7, ncol=3)

    risk_ax.axhspan(0, 25, color="#86efac", alpha=0.25, label="0–25% low")
    risk_ax.axhspan(25, 50, color="#fde68a", alpha=0.25, label="25–50% noticeable")
    risk_ax.axhspan(50, 75, color="#fdba74", alpha=0.25, label="50–75% high")
    risk_ax.axhspan(75, 100, color="#fca5a5", alpha=0.25, label="75–100% critical")

    plotted_risks = 0
    for obj in chart_targets:
        obj_diagnostics = diagnostics["objects"].get(obj)
        if obj_diagnostics is None:
            continue
        _, moon_risk = interpolate_for_chart(
            hours_local, obj_diagnostics["moon_risk"], clip=(0, 100)
        )
        _, moon_separation = interpolate_for_chart(
            hours_local, obj_diagnostics["moon_separation"], clip=(0, 180)
        )
        max_index = int(np.argmax(moon_risk))
        max_risk = moon_risk[max_index]
        max_separation = moon_separation[max_index]
        risk_ax.plot(
            chart_times,
            moon_risk,
            linewidth=2,
            label=(
                f"{short_object_name(obj, 24)} | max {max_risk:.0f}% | "
                f"Moon sep. {max_separation:.0f}°"
            ),
        )
        risk_ax.scatter(
            chart_times[max_index], max_risk, marker="x", s=55, linewidths=2
        )
        plotted_risks += 1

    if not plotted_risks:
        risk_ax.text(
            0.5,
            0.55,
            "The Moonlight penalty does not apply to the selected targets",
            transform=risk_ax.transAxes,
            ha="center",
            va="center",
        )

    risk_ax.axvline(
        current_local,
        color="red",
        linestyle="--",
        linewidth=1.5,
        label="Current time",
    )
    risk_ax.set_title("Interpolated Moonlight interference risk for the selected DSOs")
    risk_ax.set_ylabel("Moonlight impact (%)")
    risk_ax.set_ylim(0, 100)
    risk_ax.grid(True, alpha=0.3)
    risk_ax.legend(loc="upper left", fontsize=8, ncol=2)
    for axis in (probability_ax, altitude_ax, risk_ax):
        axis.set_xlim(chart_times[0], chart_times[-1])
    risk_ax.xaxis.set_major_locator(
        mdates.HourLocator(interval=1, tz=hours_local.dt.tz)
    )
    risk_ax.xaxis.set_major_formatter(
        mdates.DateFormatter("%H:%M\n%d.%m", tz=hours_local.dt.tz)
    )
    figure.tight_layout()
    plt.show()
    plt.close(figure)


def _sector_planner_render_details():
    snapshot = globals().get("_sector_planner_snapshot")
    with _sector_detail_out:
        clear_output(wait=True)
        if snapshot is None:
            print("Calculate the sector first.")
            return
        if not _sector_planner_selected:
            print("Select targets on the left and add them for detailed analysis.")
            return
        target_data = {
            name: _sector_planner_target_arrays(snapshot, name)
            for name in _sector_planner_selected
        }
        target_data = {name: data for name, data in target_data.items() if data is not None}
        if not target_data:
            print("No cached data are available for the selected targets.")
            return

        local_times = snapshot["local_times"]
        diagnostics = calculate_light_pollution_diagnostics(
            snapshot["settings"]["lat"],
            snapshot["settings"]["lon"],
            snapshot["weather"],
            list(target_data),
        )
        chart_results = {
            name: {"alt": data["altitude"]} for name, data in target_data.items()
        }
        probability_results = {
            name: data["probability"] for name, data in target_data.items()
        }
        current_local = datetime.now(timezone.utc).astimezone(
            ZoneInfo(snapshot["weather"].attrs.get("display_timezone", "UTC"))
        )
        if len(target_data) <= 5:
            _sector_planner_plot_light_pollution_chart(
                local_times,
                probability_results,
                chart_results,
                diagnostics,
                list(target_data),
                snapshot["settings"]["sun_limit"],
                current_local,
                snapshot["settings"],
                snapshot["settings"]["threshold"],
            )
        else:
            for name in target_data:
                _sector_planner_plot_light_pollution_chart(
                    local_times,
                    probability_results,
                    chart_results,
                    diagnostics,
                    list(target_data),
                    snapshot["settings"]["sun_limit"],
                    current_local,
                    snapshot["settings"],
                    snapshot["settings"]["threshold"],
                    object_filter=name,
                )

        weather = snapshot["weather"]
        cloud = _sector_planner_weather_array(weather, "cloud_cover", np.nan)
        temperature = _sector_planner_weather_array(weather, "temp", np.nan)
        humidity = _sector_planner_weather_array(weather, "humidity", np.nan)
        precipitation = _sector_planner_weather_array(weather, "precip_prob", np.nan)
        wind = _sector_planner_weather_array(weather, "wind", np.nan)
        gust = _sector_planner_weather_array(weather, "gusts", np.nan)
        moon_data = [
            f"{phase * 100:.0f}% (Alt: {altitude:.0f}°)"
            if altitude > 0
            else "Below horizon"
            for phase, altitude in zip(snapshot["moon_phase"], snapshot["moon_alt"])
        ]
        table = pd.DataFrame(
            {
                "Time (local)": local_times.dt.strftime("%H:%M %d.%m"),
                "Cloud cover (%)": [f"{value:.0f}%" for value in cloud],
                "Temp. (°C)": [f"{value:.1f}" for value in temperature],
                "Humidity (%)": [f"{value:.0f}%" for value in humidity],
                "Precipitation": [f"{value:.0f}%" for value in precipitation],
                "Wind (m/s)": [
                    f"{average:.1f} (up to {maximum:.1f})"
                    for average, maximum in zip(wind, gust)
                ],
                "Moon": moon_data,
                "Sun | Altitude": [f"{value:.1f}°" for value in snapshot["sun_alt"]],
            }
        )
        any_target_inside = np.zeros(len(table), dtype=bool)
        for name, data in target_data.items():
            table[f"{name} | Altitude"] = [f"{value:.1f}°" for value in data["altitude"]]
            table[f"{name} | Azimuth"] = [f"{value:.1f}°" for value in data["azimuth"]]
            table[f"{name} | In sector"] = np.where(data["inside"], "Yes", "No")
            table[f"{name} | Sector probability"] = [
                f"{chance:.0f}% ({reason})"
                for chance, reason in zip(data["probability"], data["reasons"])
            ]
            any_target_inside |= data["inside"]
        table = table.loc[any_target_inside].reset_index(drop=True)
        if table.empty:
            print("There are no hours when at least one selected target is inside the sector.")
        display(table.style.set_properties(**{"text-align": "left"}))


def _sector_planner_render_summary(snapshot):
    settings = snapshot["settings"]
    records = _sector_planner_records_for_mode(snapshot)
    right = (settings["direction"] + settings["half_width"]) % 360
    left = (settings["direction"] - settings["half_width"]) % 360
    with _sector_summary_out:
        clear_output(wait=True)
        print(
            f"Sector: {settings['direction']:.1f}° ± {settings['half_width']:.1f}° "
            f"({left:.1f}–{right:.1f}°), altitude {settings['alt_low']:.1f}–"
            f"{settings['alt_high']:.1f}°."
        )
        print(
            f"Targets available in the current mode: {len(records)}. "
            f"Calculated in {snapshot['elapsed'] * 1000:.1f} ms. "
            "Weather data are cached in the snapshot."
        )


def _sector_planner_refresh_from_cache(*_):
    snapshot = globals().get("_sector_planner_snapshot")
    if snapshot is None:
        return
    _sector_planner_render_summary(snapshot)
    _sector_planner_render_top3(snapshot)
    _sector_planner_update_available()


def _sector_planner_mark_details_pending(message=None):
    with _sector_detail_out:
        clear_output(wait=True)
        print(
            message
            or "Selection prepared. Click 'CALCULATE SESSION (24 hours)' "
            "to build the charts and table."
        )


def _sector_planner_validate_settings():
    if _sector_alt_low.value > _sector_alt_high.value:
        raise ValueError("The lower altitude limit cannot exceed the upper altitude limit.")
    return {
        "lat": float(_sector_lat.value),
        "lon": float(_sector_lon.value),
        "min_alt": float(_sector_min_alt.value),
        "sun_limit": float(_sector_sun_limit.value),
        "threshold": float(_sector_threshold.value),
        "direction": float(_sector_direction.value) % 360.0,
        "alt_low": float(_sector_alt_low.value),
        "alt_high": float(_sector_alt_high.value),
        "half_width": float(_sector_half_width.value),
    }


def _sector_planner_snapshot_matches_settings(snapshot, settings):
    if snapshot is None:
        return False
    saved = snapshot["settings"]
    return all(np.isclose(saved[key], settings[key]) for key in settings)


def _sector_planner_calculate(_button=None, generation=None):
    global _sector_planner_snapshot
    if generation != _sector_planner_generation:
        return
    _sector_calculate.disabled = True
    with _sector_summary_out:
        clear_output(wait=True)
        print("Downloading weather and calculating the sector snapshot...")
    try:
        settings = _sector_planner_validate_settings()
        weather = get_24h_weather(settings["lat"], settings["lon"])
        _sector_planner_snapshot = _sector_planner_build_snapshot(settings, weather)
        records = _sector_planner_snapshot["records"]
        available_names = set(records["Name"]) if "Name" in records else set()
        _sector_planner_selected[:] = [
            name for name in _sector_planner_selected if name in available_names
        ]
        _sector_planner_refresh_from_cache()
        _sector_planner_mark_details_pending(
            "The available-target list has been refreshed. Select targets and click "
            "'CALCULATE SESSION (24 hours)'."
        )
    except Exception as error:
        with _sector_summary_out:
            clear_output(wait=True)
            print(f"Sector calculation failed: {error}")
    finally:
        _sector_calculate.disabled = False


def _sector_planner_run_details(_button=None, generation=None):
    if generation != _sector_planner_generation:
        return
    _sector_run_details.disabled = True
    try:
        snapshot = globals().get("_sector_planner_snapshot")
        if snapshot is None:
            _sector_planner_mark_details_pending(
                "Click 'Refresh targets available in sector' first."
            )
            return
        settings = _sector_planner_validate_settings()
        if not _sector_planner_snapshot_matches_settings(snapshot, settings):
            _sector_planner_mark_details_pending(
                "Settings have changed. Refresh the targets available in the sector first."
            )
            return
        _sector_planner_render_details()
    except Exception as error:
        _sector_planner_mark_details_pending(f"Unable to build the analysis: {error}")
    finally:
        _sector_run_details.disabled = False


def _sector_planner_add_selected(_button=None, generation=None):
    if generation != _sector_planner_generation:
        return
    for name in _sector_available.value:
        if name not in _sector_planner_selected:
            _sector_planner_selected.append(name)
    _sector_planner_update_available()
    _sector_planner_mark_details_pending()


def _sector_planner_remove_selected(_button=None, generation=None):
    if generation != _sector_planner_generation:
        return
    remove = set(_sector_chosen.value)
    _sector_planner_selected[:] = [
        name for name in _sector_planner_selected if name not in remove
    ]
    _sector_planner_update_available()
    _sector_planner_mark_details_pending()


def _sector_planner_add_top3(_button=None, generation=None):
    if generation != _sector_planner_generation:
        return
    snapshot = globals().get("_sector_planner_snapshot")
    if snapshot is None:
        return
    for name in _sector_planner_top3(snapshot)["Name"]:
        if name not in _sector_planner_selected:
            _sector_planner_selected.append(name)
    _sector_planner_update_available()
    _sector_planner_mark_details_pending()


_sector_lat = widgets.BoundedFloatText(
    min=-90, max=90, description="Latitude:", layout=widgets.Layout(width="300px")
)
_sector_lon = widgets.BoundedFloatText(
    min=-180, max=180, description="Longitude:", layout=widgets.Layout(width="300px")
)
_sector_min_alt = widgets.BoundedFloatText(
    min=0, max=89.9, description="Min. altitude (°):", layout=widgets.Layout(width="300px")
)
_sector_sun_limit = widgets.BoundedFloatText(
    min=-90, max=90, description="Max. Sun altitude (°):", layout=widgets.Layout(width="300px")
)
_sector_threshold = widgets.IntSlider(
    value=50,
    min=10,
    max=90,
    step=5,
    description="Calendar threshold (%):",
    style={"description_width": "initial"},
    layout={"width": "400px"},
)
_sector_direction = widgets.BoundedFloatText(
    value=281, min=0, max=360, step=1, description="Direction:", layout=widgets.Layout(width="250px")
)
_sector_alt_low = widgets.BoundedFloatText(
    value=15, min=0, max=90, step=1, description="Altitude from:", layout=widgets.Layout(width="250px")
)
_sector_alt_high = widgets.BoundedFloatText(
    value=55, min=0, max=90, step=1, description="Altitude to:", layout=widgets.Layout(width="250px")
)
_sector_half_width = widgets.BoundedFloatText(
    value=10, min=0, max=90, step=1, description="Half-width:", layout=widgets.Layout(width="250px")
)
_sector_target_mode = widgets.Dropdown(
    options=["DSO", "Planets + Moon", "All"],
    value="DSO",
    description="Targets:",
    layout=widgets.Layout(width="250px"),
)
_sector_calculate = widgets.Button(
    description="Refresh targets available in sector",
    button_style="primary",
    layout=widgets.Layout(width="400px"),
)
_sector_run_details = widgets.Button(
    description="CALCULATE SESSION (24 hours)",
    button_style="success",
    layout=widgets.Layout(width="400px"),
)
_sector_add_top3 = widgets.Button(description="Add top 3 to selection", button_style="info")
_sector_search = widgets.Text(description="Search:", placeholder="For example: M31, NGC7000, Mars...")
_sector_available = widgets.SelectMultiple(
    options=[], description="Available:", layout={"height": "150px"}
)
_sector_chosen = widgets.SelectMultiple(
    options=[], description="Selected:", layout={"height": "150px"}
)
_sector_add = widgets.Button(description="Add ->", button_style="info")
_sector_remove = widgets.Button(description="<- Remove", button_style="warning")
_sector_summary_out = widgets.Output()
_sector_top3_out = widgets.Output()
_sector_detail_out = widgets.Output()

_sector_planner_links = [
    widgets.link((lat_input, "value"), (_sector_lat, "value")),
    widgets.link((lon_input, "value"), (_sector_lon, "value")),
    widgets.link((min_alt_input, "value"), (_sector_min_alt, "value")),
    widgets.link((sun_limit_input, "value"), (_sector_sun_limit, "value")),
    widgets.link((ics_threshold_input, "value"), (_sector_threshold, "value")),
]


def _sector_planner_bind_click(widget, callback):
    wrapper = lambda button, cb=callback, generation=_sector_planner_generation: cb(
        button, generation
    )
    widget.on_click(wrapper)
    _sector_planner_callbacks.append((widget, wrapper))


def _sector_planner_bind_observe(widget, callback, trait="value"):
    wrapper = lambda change, cb=callback, generation=_sector_planner_generation: (
        cb(change) if generation == _sector_planner_generation else None
    )
    widget.observe(wrapper, names=trait)
    _sector_planner_observers.append((widget, wrapper, trait))


_sector_planner_bind_click(_sector_calculate, _sector_planner_calculate)
_sector_planner_bind_click(_sector_run_details, _sector_planner_run_details)
_sector_planner_bind_click(_sector_add, _sector_planner_add_selected)
_sector_planner_bind_click(_sector_remove, _sector_planner_remove_selected)
_sector_planner_bind_click(_sector_add_top3, _sector_planner_add_top3)
_sector_planner_bind_observe(_sector_search, _sector_planner_update_available)
_sector_planner_bind_observe(_sector_target_mode, _sector_planner_refresh_from_cache)

_sector_planner_mark_details_pending(
    "Refresh the targets available in the sector, then select targets for calculation."
)

_sector_planner_ui = widgets.VBox(
    [
        widgets.HTML("<b>1. Coordinates and equipment limits</b>"),
        widgets.HBox([_sector_lat, _sector_lon]),
        widgets.HBox([_sector_sun_limit, _sector_min_alt]),
        _sector_threshold,
        widgets.HTML("<b>1.1. Capture window limits</b>"),
        widgets.HBox([_sector_direction, _sector_half_width]),
        widgets.HBox([_sector_alt_low, _sector_alt_high]),
        _sector_target_mode,
        _sector_calculate,
        _sector_summary_out,
        widgets.HTML("<hr><b>2. Top 3 sector recommendations</b>"),
        _sector_top3_out,
        _sector_add_top3,
        widgets.HTML("<hr><b>3. Target selection (available in the sector during the next 24 hours)</b>"),
        widgets.HBox(
            [
                widgets.VBox([_sector_search, _sector_available]),
                widgets.VBox(
                    [_sector_add, _sector_remove],
                    layout=widgets.Layout(justify_content="center"),
                ),
                _sector_chosen,
            ]
        ),
        widgets.HTML("<hr>"),
        _sector_run_details,
        _sector_detail_out,
    ]
)
display(_sector_planner_ui)
